# General code (for Introduction Evaluation)

### Loading the data

In [ ]:
from Code import load_documents

# replace data_path with your own path to data
data_path = r"C:\Users\Asus\Documents\GIT\LLM-Research-Judge\DATA\Test"
data_dict = load_documents(data_path)

In [ ]:
# [OPTIONAL]: Use the code snippet below to know your token count before processing
# If you changed file_content_header in the load_documents function, 
# replace 'file_content' with your own file_content_header name.

from Code import token_count

data_dict['input_token_count'] = [
    token_count(content) for content in data_dict['file_content']
    ]

print(f"\nList of tokens per each input: {data_dict['input_token_count']}")
print(f"Sum of tokens for all inputs: {sum(data_dict['input_token_count'])}")

### Critera and prompt processing and formatting

In [ ]:
from Code import (introduction_criteria_definer, 
                  token_count)


format_instructions = introduction_criteria_definer().get_format_instructions()


introduction_prompt = """
You are a medical doctor, academic researcher, and experienced peer reviewer.\
Critically and strictly evaluate the following Introduction section of a scientific paper: 
{intro_text}

Use the given criteria to evaluate the introduction.
Only return a JSON object that complies to the criteria below.
{format_instructions}
"""

Token count estimation (optional)

In [ ]:
total_tokens = token_count(introduction_prompt) + token_count(format_instructions)

print(f'''Your toekn count is as follows:
- main_prompt: {token_count(introduction_prompt)}
- format_instructions: {token_count(format_instructions)}
- total input tokens (excluding the tokens of your input files): {total_tokens}
''')

### LLM evaluation and response generation

In [ ]:
from Code import introduction_criteria_definer
from Code import openai_response_generator

parser = introduction_criteria_definer()

# Uncomment the code below to use openai (paid) LLM response
results = openai_response_generator(data=data_dict,
                                 prompt=introduction_prompt,
                                 parser=parser,
                                 verbose=True,
                                 model='gpt-4.1-mini')

# Uncomment the code below to use openrouter (free & paid) LLM response
# results = openrouter_llm_response(data=data_dict,
#                                  prompt=main_prompt,
#                                  parser=parser,
#                                  verbose=True,
#                                  model='huggingfaceh4/zephyr-7b-beta:free')

print(results)


### Saving the results

In [ ]:
from Code import csv_output
csv_output(results, csv_file_name='test.csv')

# General Code for Prof. Alborzi Congress 2025

### Data Loading

In [ ]:
from Code.robust_abstract_parser import excel_to_json

input_path = r'.\ABS_DATA\prof_alborzi_new.xlsx'
output_filename = 'prof_alborzi_new.json'

results = excel_to_json(input_path, output_filename=output_filename)

### Initial Screening

In [ ]:
from Code.screen import abstract_screen_prof_alborzi

screen_results = abstract_screen_prof_alborzi(output_filename)

In [ ]:
import pandas as pd
import json

with open('screening_results_new.json', encoding='utf-8') as f:
    json_file = json.load(f)

df = pd.DataFrame.from_dict(json_file)
df.sort_values(by='abstract_code', inplace=True, ascending=True)
df_trimmed = df[df['accurate_study_type'] == False]
df_trimmed.to_csv('screening_results_case_report.csv', encoding='utf-8')
df_trimmed.to_json('screening_results_case_report.json', orient='records', indent=2)

### Full Evaluation

In [ ]:
from Code.generator import prof_alborzi_congress_eval_async
from Code.enhanced_schemas import abstract_criteria_definer

prompt_abs = """You are a highly experienced clinician, academic researcher, and peer reviewer with knowledge background in infectious diseases, microbiology, and epidemiology.
Critically, objectively, and strictly evaluate the following Abstract sent for an international Infectious Disease and Microbiology Congres. The congress theme is 'Guidelines and Innovations in Clinical Microbiology and Infectious Diseases'.

Abstract Text: {abs_text}

Use the given criteria to evaluate the abstract.
Only return a JSON object that complies to the criteria below:

{format_instructions}
"""

parser = abstract_criteria_definer()

llm_eval = await prof_alborzi_congress_eval_async(data_path='not_narrative.json',
                                 prompt=prompt_abs,
                                 parser=parser,
                                 model='gpt-5-mini'
                                 )

In [ ]:
from Code.utils import json_to_excel

json_path = r'C:\Users\Asus\Documents\GIT\LLM-as-a-Research-Judge\LLM_outputs\LLM_Output_gpt-5-mini_not_narrative.json'
json_to_excel(json_path=json_path)

# General code (Abstract)

### Data parsing

In [ ]:
from Code import process_abs_data_folder_corrected

dir_path = r'.\ABS_DATA\Congress_Booklets'
output_dir = r'.\parse_results'
parse_results = process_abs_data_folder_corrected(folder_path=dir_path,
                output_dir=output_dir)

### Abstract screening (before full evaluation)

In [ ]:
from Code import abstract_screening_results

input_dir = r'C:\Users\Asus\Documents\GIT\LLM-as-a-Research-Judge\ABS_DATA\Test\Featured'
data_dict = abstract_screening_results(input_dir, 
                                         evaluate_criteria=True)

### Testing the performance of each schema

In [ ]:
from pathlib import Path

from Code.complete_workflow import run_complete_evaluation

path_to_file = r'C:\Users\Asus\Documents\GIT\LLM-as-a-Research-Judge\ABS_DATA\Test\Full_test_parsed.json'
try:
    results = run_complete_evaluation(
        parsed_abstracts_file=path_to_file,
        output_dir='test_eval_results',
        model='gpt-4.1-mini',
        max_abstracts=2  # small for testing
    )
    print("✓ Complete evaluation succeeded!")

except Exception as e:
    print(f"✗ Complete evaluation failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
from Code.split_evaluation_workflow import compare_evaluation_methods

path_to_file = r'C:\Users\Asus\Documents\GIT\LLM-as-a-Research-Judge\ABS_DATA\Test\Full_test_parsed.json'

compare_evaluation_methods(path_to_file, max_abstracts=10)


### Critera and prompt processing and formatting

In [ ]:
from Code import (abstract_criteria_definer, 
                  token_count)

format_instructions = abstract_criteria_definer().get_format_instructions()


abstract_prompt = """
You are a highly experienced academic researcher and peer reviewer \
with knowledge background in infectious diseases, microbiology, and epidemiology.\
Critically, objectively, and strictly evaluate the following Abstract sent for an international Infectious Disease and Microbiology Congress:

{intro_text}

Use the given criteria to evaluate the introduction.
Only return a JSON object that complies to the criteria below.
{format_instructions}
"""

token count (optional)

In [ ]:
# [OPTIONAL]: Uncomment the code snippet below to know your token count before processing
# In your_prompt, define if you are using introduction_prompt or abstract_prompt

total_tokens = token_count(abstract_prompt) + token_count(format_instructions)

print(f'''Your toekn count is as follows:
- main_prompt: {token_count(abstract_prompt)}
- format_instructions: {token_count(format_instructions)}
- total input tokens (excluding the tokens of your input files): {total_tokens}
''')

### LLM evaluation and response generation

In [ ]:
from Code import abstract_criteria_definer
from Code import openai_response_generator, openrouter_llm_response

parser = abstract_criteria_definer()

final_data_dict = data_dict[data_dict['screening_status'] == 'Accepted']

# Uncomment the code below to use openai (paid) LLM response
results = openai_response_generator(data=final_data_dict,
                                 prompt=abstract_prompt,
                                 parser=parser,
                                 verbose=True,
                                 model='gpt-4.1-mini')

# Uncomment the code below to use openrouter (free & paid) LLM response
# results = openrouter_llm_response(data=final_data_dict,
#                                  prompt=main_prompt,
#                                  parser=parser,
#                                  verbose=True,
#                                  model='huggingfaceh4/zephyr-7b-beta:free')

print(results)


### Saving the results

In [ ]:
from Code import csv_output
csv_output(results, csv_file_name='test.csv')